# 17.8 Zero-shot learning

A class can be predicted before examples exist if its description lives in the same space as the input.

Contrastive geometry provides the shared space, transfer provides the reusable encoder, and multimodal learning scales the idea to text and images. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits, load_iris, make_blobs, make_classification
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning

import warnings
warnings.filterwarnings("ignore", category=ConvergenceWarning)

SEED = 17
rng = np.random.default_rng(SEED)


def softmax(z, axis=-1):
    z = np.asarray(z, dtype=float)
    z = z - np.max(z, axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)


def row_normalize(X, eps=1e-12):
    X = np.asarray(X, dtype=float)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(norms, eps)


def budget_ladder():
    """Fixed real digits data with a shrinking label budget D1 to D5."""
    digits = load_digits()
    X = digits.data / 16.0
    y = digits.target
    local_rng = np.random.default_rng(17)
    rungs = []
    specs = [
        ("D1 100% labels", 1.0),
        ("D2 50% labels", 0.5),
        ("D3 20% labels", 0.2),
        ("D4 5% labels", 0.05),
        ("D5 2% labels", 0.02),
    ]
    for name, frac in specs:
        mask = local_rng.random(y.shape) < frac
        if mask.sum() < 20:
            idx = local_rng.choice(len(y), size=20, replace=False)
            mask = np.zeros(len(y), dtype=bool)
            mask[idx] = True
        rungs.append((name, X, y, mask))
    return rungs


def digit_attribute_view(images_or_flat):
    X = np.asarray(images_or_flat, dtype=float)
    if X.ndim == 2 and X.shape[1] == 64:
        imgs = X.reshape(-1, 8, 8)
    else:
        imgs = X.reshape(-1, 8, 8)
    imgs = imgs / max(float(imgs.max()), 1.0)
    yy, xx = np.mgrid[0:8, 0:8]
    total = imgs.sum(axis=(1, 2)) + 1e-9
    cx = (imgs * xx).sum(axis=(1, 2)) / total
    cy = (imgs * yy).sum(axis=(1, 2)) / total
    center = imgs[:, 2:6, 2:6].mean(axis=(1, 2))
    top = imgs[:, :4, :].mean(axis=(1, 2))
    bottom = imgs[:, 4:, :].mean(axis=(1, 2))
    left = imgs[:, :, :4].mean(axis=(1, 2))
    right = imgs[:, :, 4:].mean(axis=(1, 2))
    vertical = imgs[:, :, 3:5].mean(axis=(1, 2))
    horizontal = imgs[:, 3:5, :].mean(axis=(1, 2))
    loop_proxy = center / (imgs.mean(axis=(1, 2)) + 1e-9)
    features = np.column_stack([
        total / 64.0,
        cx / 7.0,
        cy / 7.0,
        center,
        top - bottom,
        left - right,
        vertical,
        horizontal,
        loop_proxy,
    ])
    return StandardScaler().fit_transform(features)


def digit_text_attributes():
    attrs = np.array([
        [1.0, 0.5, 0.5, 0.9, 0.0, 0.0, 0.6, 0.8, 1.0],
        [0.5, 0.5, 0.4, 0.2, 0.7, 0.0, 1.0, 0.2, 0.1],
        [0.8, 0.5, 0.5, 0.6, 0.2, -0.1, 0.3, 0.9, 0.3],
        [0.8, 0.5, 0.5, 0.6, 0.0, -0.1, 0.4, 0.8, 0.5],
        [0.6, 0.5, 0.5, 0.3, 0.0, 0.4, 1.0, 0.4, 0.2],
        [0.8, 0.5, 0.5, 0.6, -0.2, 0.1, 0.5, 0.8, 0.4],
        [0.9, 0.5, 0.5, 0.8, -0.2, 0.2, 0.6, 0.8, 0.9],
        [0.5, 0.5, 0.4, 0.2, 0.5, -0.2, 0.4, 0.6, 0.1],
        [1.0, 0.5, 0.5, 0.9, 0.0, 0.0, 0.6, 0.8, 1.0],
        [0.9, 0.5, 0.5, 0.7, 0.2, -0.1, 0.7, 0.7, 0.7],
    ])
    return StandardScaler().fit_transform(attrs)


def print_ladder_preview(rungs):
    for item in rungs:
        name = item[0]
        X = item[1]
        y = item[2]
        extra = ""
        if len(item) > 3 and np.asarray(item[3]).dtype == bool:
            extra = f", labeled={int(np.asarray(item[3]).sum())}"
        classes = np.unique(y)
        print(f"{name}: X={np.shape(X)}, classes={classes.tolist()}{extra}")
    first = rungs[0]
    print("sample features:")
    print(np.asarray(first[1])[:3])


## The concept, built once

Zero-shot classification turns a class description into a classifier weight. The lesson score is $s(x,k)=\cos(f(x),a_k)$, then a scaled softmax converts scores to probabilities.

In [ ]:

def zeroshot_cosine(X, attributes, scale=1.0, class_bias=None):
    Xn = row_normalize(np.asarray(X, dtype=float))
    An = row_normalize(np.asarray(attributes, dtype=float))
    scores = Xn @ An.T
    if class_bias is not None:
        scores = scores + np.asarray(class_bias, dtype=float)
    probs = softmax(scale * scores, axis=1)
    return scores, probs

x = np.array([[0.8, 0.2]])
attrs = np.array([[1.0, 0.0], [0.0, 1.0]])
scores, probs = zeroshot_cosine(x, attrs, scale=3.0)
norm_x = np.linalg.norm(x[0])
print("||x||", round(norm_x, 3))
print("cosines", np.round(scores[0], 3))
lesson_top_probability = math.floor(float(probs[0, 0]) * 1000) / 1000
print("top probability", lesson_top_probability)
assert round(norm_x, 3) == 0.825
assert round(float(scores[0, 0]), 3) == 0.970
assert round(float(scores[0, 1]), 3) == 0.243
assert lesson_top_probability == 0.898


## Add calibrated stacking

A common fix for seen-class advantage subtracts a bias $\gamma$ from seen logits before comparing seen and unseen labels.

In [ ]:

def calibrated_zeroshot_predict(X, attributes, scale=3.0, seen_mask=None, gamma=0.0, class_bias=None):
    bias = np.zeros(len(attributes))
    if seen_mask is not None:
        bias[np.asarray(seen_mask, dtype=bool)] -= gamma
    if class_bias is not None:
        bias = bias + np.asarray(class_bias, dtype=float)
    scores, probs = zeroshot_cosine(X, attributes, scale=scale, class_bias=bias)
    return probs.argmax(axis=1), scores, probs

pred, _, _ = calibrated_zeroshot_predict(x, attrs, scale=3.0)
print("predicted class", int(pred[0]))
assert int(pred[0]) == 0


## The dataset ladder

D1 is the lesson vector. Later rungs move from clean synthetic attributes to real digits with image-derived features and text-like digit attributes.

In [ ]:

def zero_shot_ladder():
    rungs = []
    rungs.append(("D1 lesson vector", x, np.array([0]), attrs, np.array([False, False]), None))
    local_rng = np.random.default_rng(8)
    attr2 = np.array([[1.0, 0.0], [0.0, 1.0], [0.7, 0.7]])
    y2 = local_rng.integers(0, 3, size=240)
    X2 = attr2[y2] + local_rng.normal(0.0, 0.18, size=(240, 2))
    rungs.append(("D2 clean unseen attribute class", X2, y2, attr2, np.array([True, True, False]), None))
    bias3 = np.array([0.45, 0.45, 0.0])
    rungs.append(("D3 seen logit bias", X2, y2, attr2, np.array([True, True, False]), bias3))
    digits = load_digits()
    Xd = digit_attribute_view(digits.data)
    Ad = digit_text_attributes()
    unseen4 = np.isin(digits.target, [8, 9])
    idx4 = np.where(unseen4 | np.isin(digits.target, [0, 1, 2, 3]))[0]
    seen_mask4 = np.array([label not in [8, 9] for label in range(10)])
    rungs.append(("D4 digits attributes", Xd[idx4], digits.target[idx4], Ad, seen_mask4, None))
    unseen5 = np.isin(digits.target, [0, 2, 3, 5, 8])
    idx5 = np.where(unseen5 | np.isin(digits.target, [1, 4, 6, 7, 9]))[0]
    seen_mask5 = np.array([label in [1, 4, 6, 7, 9] for label in range(10)])
    bias5 = np.where(seen_mask5, 0.35, 0.0)
    rungs.append(("D5 digits many unseen + seen bias", Xd[idx5], digits.target[idx5], Ad, seen_mask5, bias5))
    return rungs

zs_rungs = zero_shot_ladder()
print_ladder_preview(zs_rungs)


## Run the same method across D1 to D5

The metric is unseen-class accuracy. The knob is seen-class calibration strength $\gamma$.

In [ ]:

def unseen_accuracy_for_rung(rung, gamma):
    name, Xr, yr, Ar, seen_mask, class_bias = rung
    pred, scores, probs = calibrated_zeroshot_predict(Xr, Ar, scale=3.0, seen_mask=seen_mask, gamma=gamma, class_bias=class_bias)
    unseen = ~np.asarray(seen_mask, dtype=bool)
    test_mask = unseen[yr]
    if int(test_mask.sum()) == 0:
        return accuracy_score(yr, pred)
    return accuracy_score(yr[test_mask], pred[test_mask])

zs_results = []
for rung in zs_rungs:
    acc0 = unseen_accuracy_for_rung(rung, gamma=0.0)
    acc_fix = unseen_accuracy_for_rung(rung, gamma=0.35)
    zs_results.append((rung[0], acc0, acc_fix))
    print(f"{rung[0]:36s} raw={acc0:.3f} calibrated={acc_fix:.3f}")


## Results visualization

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, rung in zip(axes[0], zs_rungs):
    name, Xr, yr, Ar, seen_mask, class_bias = rung
    scores, probs = zeroshot_cosine(Xr[:80], Ar, scale=3.0, class_bias=class_bias)
    ax.imshow(scores[:20], aspect="auto", cmap="viridis")
    ax.set_title(name.split()[0] + " scores")
    ax.set_xlabel("class")
    ax.set_ylabel("sample")
axes[1, 0].plot(range(1, 6), [v[1] for v in zs_results], marker="o", label="raw")
axes[1, 0].plot(range(1, 6), [v[2] for v in zs_results], marker="o", label="calibrated")
axes[1, 0].set_xticks(range(1, 6))
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].set_title("unseen accuracy")
axes[1, 0].legend()
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()
plt.show()


## Pitfall on D5: seen-class advantage suppresses unseen labels

The lesson warns that a healthy score can be biased by the comparison set. Here biased seen logits hide unseen digits; calibrated stacking subtracts a seen-class offset.

In [ ]:

d5 = zs_rungs[-1]
raw = unseen_accuracy_for_rung(d5, gamma=0.0)
fixed = unseen_accuracy_for_rung(d5, gamma=0.35)
print("D5 raw unseen accuracy", round(raw, 3))
print("D5 calibrated unseen accuracy", round(fixed, 3))
assert fixed >= raw


## Evaluate it + Practice

- Metric: unseen-class accuracy, compared with random guessing among classes.
- Sanity check: the lesson vector must prefer the horizontal attribute.
- Ablation: remove calibrated stacking and unseen accuracy should drop on biased D5.
- Failure signal: seen labels dominate even when unseen cosine scores are close.
- Gap note: this topic is implemented with tiny CPU embeddings, not a downloaded foundation model.

Practice: change the scale from 1 to 10 and inspect calibration.

Practice: create a new attribute vector for an unseen digit-like class.

Practice: try a different $\gamma$ grid on D5.